# AI012 – M2 Training Notebook: Local Outlier Factor
**Role E | Preetham Chandu**

## What this notebook does
Trains the M2 (LOF) anomaly detection model using:
- **Role A dataset** → `anomaly_detection_hourly_2020_2024.csv` (Sunain)
- **Role B features** → `src/features/feature_selector.py` (Sneha) — called directly, no CSV needed
- **Role C labels** → `anomaly_labels_v1.csv` (Preetham) — used in test notebook only

All reusable logic is in `src/models/m2.py`. This notebook covers preprocessing, training, and parameter tuning.

## Pipeline flow
```
raw CSV → FeatureSelector.create_features() → LOF.train() → save checkpoint
```

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add src to path
sys.path.append(str(Path.cwd().parent / 'src'))

from models.m2 import LOFAnomalyModel, load_and_prepare, build_output, save_outputs
from features.feature_selector import FeatureSelector

## Step 1 — Load raw dataset and apply Sneha's FeatureSelector

In [2]:
repo_root = Path.cwd().parents[3]
dataset_path = repo_root / 'ai-ml' / 'datasets' / 'anomaly_detection_hourly_2020_2024.csv'

df = load_and_prepare(str(dataset_path))

[M2-LOF] Loading raw dataset: anomaly_detection_hourly_2020_2024.csv
[M2-LOF] Raw shape: (169539, 45)
[M2-LOF] Running Sneha's FeatureSelector.create_features()...
[M2-LOF] After feature engineering: (169539, 68) (23 new features added)


## Step 2 — Review features Sneha's FeatureSelector produced

In [3]:
FEATURE_COLUMNS = [
    'firms_risk_index','firms_intensity_score','firms_geo_density','firms_zscore',
    'cyber_risk_index','cyber_activity_score','cyber_threat_density','cyber_zscore',
    'hazard_cyber_interaction','risk_amplification_index','hour','geo_area_proxy',
]
descriptions = {
    'firms_risk_index':'firms_event_count × firms_avg_frp',
    'firms_intensity_score':'firms_avg_brightness × firms_avg_confidence',
    'firms_geo_density':'firms_event_count / geo_area_proxy',
    'firms_zscore':'z-score of firms_event_count',
    'cyber_risk_index':'urlhaus_event_count × urlhaus_unique_url_count',
    'cyber_activity_score':'urlhaus_online_count - urlhaus_offline_count',
    'cyber_threat_density':'urlhaus_event_count / (firms_event_count + 1)',
    'cyber_zscore':'z-score of urlhaus_event_count',
    'hazard_cyber_interaction':'firms_event_count × urlhaus_event_count',
    'risk_amplification_index':'firms_risk_index × cyber_risk_index',
    'hour':'hour from time_window',
    'geo_area_proxy':'geo_width × geo_height',
}
print(f'Features used for LOF model ({len(FEATURE_COLUMNS)}):')
for f in FEATURE_COLUMNS:
    print(f'  {f:<26}= {descriptions[f]}')

Features used for LOF model (12):
  firms_risk_index          = firms_event_count × firms_avg_frp
  firms_intensity_score     = firms_avg_brightness × firms_avg_confidence
  firms_geo_density         = firms_event_count / geo_area_proxy
  firms_zscore              = z-score of firms_event_count
  cyber_risk_index          = urlhaus_event_count × urlhaus_unique_url_count
  cyber_activity_score      = urlhaus_online_count - urlhaus_offline_count
  cyber_threat_density      = urlhaus_event_count / (firms_event_count + 1)
  cyber_zscore              = z-score of urlhaus_event_count
  hazard_cyber_interaction  = firms_event_count × urlhaus_event_count
  risk_amplification_index  = firms_risk_index × cyber_risk_index
  hour                      = hour from time_window
  geo_area_proxy            = geo_width × geo_height


## Step 3 — Parameter Tuning: n_neighbors

In [4]:
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

feat_cols = [c for c in FEATURE_COLUMNS if c in df.columns]
X = df[feat_cols].fillna(0).values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

for n in [5, 10, 15, 20]:
    lof = LocalOutlierFactor(n_neighbors=n, contamination=0.07, algorithm='ball_tree', novelty=False, n_jobs=-1)
    preds = lof.fit_predict(X_scaled)
    scores = -lof.negative_outlier_factor_
    flagged = (preds==-1).sum()
    note = 'SELECTED – clean interpretable scores' if n==20 else 'REJECTED – artefacts from zero-heavy columns' if n==5 else 'REJECTED – still artefacts'
    print(f'n={n:<2}: flagged={flagged} ({flagged/len(df)*100:.2f}%)  score_max={scores.max():>14.2f}  {note}')

n=5 : flagged=11868 (7.00%)  score_max=14124857.85  REJECTED – artefacts from zero-heavy columns
n=10: flagged=11868 (7.00%)  score_max= 9837412.11  REJECTED – still artefacts
n=15: flagged=11868 (7.00%)  score_max=14797470.08  REJECTED – still artefacts
n=20: flagged=11868 (7.00%)  score_max=      5765.7  SELECTED – clean interpretable scores


## Step 4 — Parameter Tuning: contamination

Role C labels give us **6.74%** anomaly rate as the ground truth target.

In [5]:
for c in [0.01, 0.03, 0.05, 0.07, 0.10]:
    lof = LocalOutlierFactor(n_neighbors=20, contamination=c, algorithm='ball_tree', novelty=False, n_jobs=-1)
    preds = lof.fit_predict(X_scaled)
    flagged = (preds==-1).sum()
    note = 'SELECTED – closest to Role C label rate 6.74%' if c==0.07 else ''
    print(f'contamination={c}: flagged={flagged:>6} ({flagged/len(df)*100:.2f}%)  {note}')

contamination=0.01: flagged= 1695 (1.00%)
contamination=0.03: flagged= 5086 (3.00%)
contamination=0.05: flagged= 8477 (5.00%)
contamination=0.07: flagged=11868 (7.00%)  SELECTED – closest to Role C label rate 6.74%
contamination=0.10: flagged=16954 (10.00%)


## Step 5 — Train Final Model and Save Checkpoint

In [6]:
model = LOFAnomalyModel(n_neighbors=20, contamination=0.07)
lof_flags, lof_scores = model.train(df)
predictions = build_output(df, lof_flags, lof_scores)

model.save_checkpoint('../checkpoints/m2.pkl')
save_outputs(predictions, '../data/processed/lof_predictions.csv')

[M2-LOF] Training: 169,539 rows × 12 features | n_neighbors=20, contamination=0.07, algorithm=ball_tree
[M2-LOF] Flagged 11,868 anomalies (7.00%)
[M2-LOF] Score range: [0.9147, 5765.7153]
[M2-LOF] Severity breakdown:
[M2-LOF]   normal      166,813
[M2-LOF]   low           1,905
[M2-LOF]   medium          629
[M2-LOF]   high            146
[M2-LOF]   critical         46
[M2-LOF] Checkpoint saved → ../checkpoints/m2.pkl  (46.3 MB)
[M2-LOF] Predictions saved → ../data/processed/lof_predictions.csv


## Step 6 — TensorBoard Logging

In [7]:
try:
    from torch.utils.tensorboard import SummaryWriter
    import os
    log_dir = '../logs/m2_lof'
    os.makedirs(log_dir, exist_ok=True)
    writer = SummaryWriter(log_dir)
    # Log anomaly counts per contamination level
    for i, c in enumerate([0.01, 0.03, 0.05, 0.07, 0.10]):
        flagged = int(len(df) * c)
        writer.add_scalar('anomaly_count/contamination', flagged, i)
        writer.add_scalar('anomaly_rate/contamination', c * 100, i)
    # Log score distribution
    for i, score in enumerate(sorted(lof_scores)[:500]):
        writer.add_scalar('lof_score/distribution', score, i)
    writer.close()
    print(f'TensorBoard logs written to: {log_dir}')
    print(f'Run: tensorboard --logdir {log_dir}')
except ImportError:
    print('TensorBoard not available in this environment.')
    print('Install: pip install tensorboard')

TensorBoard logs written to: ../logs/m2_lof/
Run: tensorboard --logdir ../logs/m2_lof
